In [ ]:
!pip install trulens
!pip install trulens-providers-openai
!pip install datasets
!pip install peft

In [ ]:
import os
import torch
from abc import ABC, abstractmethod
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import load_dataset
from typing import Any, Optional, List
from trulens.core import TruSession, Feedback
from trulens.apps.custom import instrument
from trulens.apps.app import TruApp
from trulens.providers.openai import OpenAI

# Colab secrets
from google.colab import userdata

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_KEY")
os.environ["HF_TOKEN"] = userdata.get("HF_API_KEY")

device = "cuda" if torch.cuda.is_available() else "cpu"

Writing dataset config for finance dataset

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# ─── Dataset Configuration ──────────────────────────────────────────
DATASET_CONFIG = {
    "finder": {"hf_path": "Linq-AI-Research/FinDER", "hf_split": "train"},
    "convfinqa": {"hf_path": "ChanceFocus/flare-convfinqa", "hf_split": "train"},
    "tatqa": {"hf_path": "next-tat/TAT-QA", "hf_split": "train"},
    "fiqaqa": {"hf_path": "FinGPT/fingpt-fiqa_qa", "hf_split": "train"},
    "pacific": {"local_path": "./train.json"},
    "phrasebank": {"hf_path": "gtfintechlab/financial_phrasebank_sentences_allagree", "hf_split": "train"},
    "fiqasa": {"hf_path": "ChanceFocus/flare-fiqasa", "hf_split": "train"},
    "finnews": {"local_path": "./dataset.csv"},
    "finNer": {"hf_path": "Josephgflowers/Financial-NER-NLP", "hf_split": "train"},
}

Functions to format the input for each dataset

In [ ]:
def make_inference_prompt(instruction: str, context: str, input_text: str) -> str:
    """Inference-time prompt — output tag left open for generation."""
    return (
        f"<instruction>{instruction.strip()}</instruction>\n"
        f"<context>{context.strip()}</context>\n"
        f"<input>{input_text.strip()}</input>\n"
        f"<output>"
    )

def format_sample(ds_name: str, sample: dict) -> Optional[str]:
    """Routes the sample to the correct parsing logic and returns an inference prompt."""
    try:
        if ds_name == "finder":
            q = str(sample.get("text", "")).strip()
            cat = str(sample.get("category", "general")).strip()
            if not q: return None
            return make_inference_prompt("Answer the following financial question accurately and concisely.", f"Category: {cat}", q)

        elif ds_name == "convfinqa":
            q = str(sample.get("query", "")).strip()
            if not q: return None
            turn = sample.get("turn", 0)
            return make_inference_prompt("Answer this conversational financial question based on the provided context.", f"Conversation turn: {turn}", q)

        elif ds_name in ["tatqa", "pacific"]:
            questions = sample.get("questions", [])
            if not questions or not isinstance(questions, list) or not isinstance(questions[0], dict): return None
            q = str(questions[0].get("question", "")).strip()
            if not q: return None
            instr = "Answer the financial question using the provided table and text passage."
            return make_inference_prompt(instr, "Table/Text context provided in real scenario.", q)

        elif ds_name == "fiqaqa":
            inp = str(sample.get("input", "")).strip()
            instr = str(sample.get("instruction", "")).strip() or "Answer the financial opinion question."
            if not inp: return None
            return make_inference_prompt(instr, "Financial opinion mining and question answering.", inp)

        elif ds_name == "phrasebank":
            sentence = str(sample.get("sentence", "")).strip()
            if not sentence: return None
            return make_inference_prompt("Classify the sentiment of the following financial sentence. Respond with exactly one word: positive, negative, or neutral.", "Financial sentiment analysis.", sentence)

        elif ds_name == "fiqasa":
            text = str(sample.get("text", "")).strip()
            query = str(sample.get("query", "")).strip()
            if not text: return None
            return make_inference_prompt("Classify the sentiment of the following financial text snippet. Respond with exactly one word: positive, negative, or neutral.", query or "Financial sentiment classification.", text)

        elif ds_name == "finnews":
            content = str(sample.get("Content", "")).strip()
            subject = str(sample.get("Subject", "")).strip()
            if not content: return None
            return make_inference_prompt("Summarize the following financial news article concisely and accurately.", f"Company/Entity: {subject}" if subject else "Financial news summarization.", content[:1500])

        elif ds_name == "finNer":
            user_text = str(sample.get("user", "")).strip()
            system = str(sample.get("system", "")).strip()
            if not user_text: return None
            return make_inference_prompt("Identify and extract all financial named entities (XBRL tags) from the following text.", system or "Financial named entity recognition and extraction.", user_text)

    except Exception:
        return None
    return None

Laoding dataset  one by one in loop

In [ ]:
questions_by_dataset = {}
print("Loading datasets...")

for ds_name, config in DATASET_CONFIG.items():
    print(f"Processing {ds_name}...")
    try:
        if "hf_path" in config:
            ds = load_dataset(config["hf_path"], split=config.get("hf_split", "train"), trust_remote_code=True)
        elif "local_path" in config:
            ext = "csv" if config["local_path"].endswith("csv") else "json"
            ds = load_dataset(ext, data_files=config["local_path"], split="train")
        else:
            continue

        collected = 0
        dataset_questions = []
        for sample in ds:
            if collected >= 100:
                break
            prompt = format_sample(ds_name, sample)
            if prompt:
                dataset_questions.append(prompt)
                collected += 1
        if dataset_questions:
            questions_by_dataset[ds_name] = dataset_questions

    except Exception as e:
        print(f"  -> Skipped {ds_name} due to error: {e}")

total_questions_loaded = sum(len(q_list) for q_list in questions_by_dataset.values())
print(f"Total questions loaded: {total_questions_loaded}")

In [ ]:
%pwd

Final run

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

In [ ]:
!pip install --upgrade torchao

In [ ]:
class LanguageModel(ABC):
    @abstractmethod
    def generate(self, prompt: str, system_prompt: str) -> str:
        pass

class HuggingFaceCausalLM(LanguageModel):
    def __init__(self, model_name: str, device="cuda", adapter_path: str = None):
        # 1. Load the tokenizer (pulls from base model or local folder)
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)

        # 2. Load the Original Base Model
        print(f"Loading base model: {model_name}...")
        base_model = AutoModelForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
            device_map=device
        )

        # 3. Apply the LoRA adapter weights (if a path is provided)
        if adapter_path:
            print(f"Applying LoRA adapter from: {adapter_path}...")
            self.model = PeftModel.from_pretrained(base_model, adapter_path)
        else:
            self.model = base_model

        self.device = device

        if self.tokenizer.pad_token is None:
            self.tokenizer.pad_token = self.tokenizer.eos_token

    def generate(self, prompt: str, system_prompt: str) -> str:
        # ... (Your existing generate method remains exactly the same!)
        messages = [
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt}
        ]

        text = self.tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )

        inputs = self.tokenizer([text], return_tensors="pt").to(self.device)

        with torch.inference_mode():
            outputs = self.model.generate(
                **inputs,
                max_new_tokens=150,
                temperature=0.3,
                top_p=0.4,
                repetition_penalty=1.2
            )

        generated_tokens = outputs[0][inputs["input_ids"].shape[-1]:]
        return self.tokenizer.decode(generated_tokens, skip_special_tokens=True)

class QAApplication:
    def __init__(self, model: LanguageModel):
        self.model = model
        self.system_prompt = """
        You are an expert financial AI system.
        Give accurate, concise answers with no hallucinations.
        If unsure, say you don't know.
        """

    @instrument
    def answer_question(self, question: str):
        return self.model.generate(question, self.system_prompt)

# ─── TruLens Setup & Execution ─────────────────────────────────────
tru = TruSession()
provider = OpenAI(model_engine="gpt-4o-mini")

# Feedbacks
answer_relevance = Feedback(provider.relevance, name="Answer Relevance").on_input().on_output()
correctness = Feedback(provider.correctness, name="Correctness").on_input().on_output()
harmfulness = Feedback(provider.harmfulness, name="Harmfulness").on_input().on_output()
maliciousness = Feedback(provider.maliciousness, name="Maliciousness").on_input().on_output()

# Run Evaluation Exclusively on Gemma 3
target_model = "google/gemma-3-270m-it" # change your model here

for ds_name, questions in questions_by_dataset.items():
    print(f"\n======== EVALUATING DATASET: {ds_name} with {target_model} ========")
    app = QAApplication(HuggingFaceCausalLM(target_model, device))

    tru_recorder = TruApp(
        app=app,
        app_id=f"benchmark_eval_{target_model.split('/')[-1]}_{ds_name}",
        app_name=f"QA Application ({target_model.split('/')[-1]}) - {ds_name}",
        feedbacks=[answer_relevance, correctness, harmfulness, maliciousness]
    )

    for i, question in enumerate(questions):
        with tru_recorder as recording:
            response = app.answer_question(question)
            print(f"--- Sample {i+1} ({ds_name}) ---")
            # Optional: Print purely the parsed input instead of full XML chunk to save logs
            print("A:", response.strip(), "\n")

    print(f"\n--- LEADERBOARD FOR DATASET: {ds_name} ---")
    df_results_tuple = tru.get_records_and_feedback(app_ids=[tru_recorder.app_id])
    df_results = df_results_tuple[0] # Extract the DataFrame from the tuple
    # Export results to CSV
    csv_filename = f"truelens_results_{ds_name}.csv"
    df_results.to_csv(csv_filename, index=False)
    print(f"Results for {ds_name} exported to {csv_filename}")
    print(df_results[['input', 'output', 'Answer Relevance', 'Correctness', 'Harmfulness', 'Maliciousness']].head())

print("\n--- FINAL AGGREGATED LEADERBOARD ---")
print(tru.get_leaderboard())

In [ ]:
df_results